# Challenge de la Sesión 11 — Mejora, prueba y decide

**Evaluación y mejora continua de modelos** · *Desarrollo y Evaluación de Modelos Propios* · Universidad del Rosario · Rosario GSB

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/11-continuous-improvement/challenge.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/11-continuous-improvement/challenge.ipynb)

| | |
|---|---|
| **Modalidad** | Individual |
| **Tiempo** | 40 minutos en clase · entrega hasta las **23:59 de hoy** (hora de Colombia) |
| **Uso de IA** | Permitido, con **registro obligatorio** al final del notebook ([política de uso de IA](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/docs/politica-uso-ia.md)). Sin registro, el componente de reflexión y registro de IA vale 0.0 |
| **Micro-sustentaciones** | Durante el challenge se eligen 3–4 estudiantes al azar para explicar su notebook en 2–3 minutos, sin IA. Si no puedes explicar tu entrega, la nota máxima de este challenge es **3.0** |
| **Rúbrica** | Ejecución técnica **40%** · Análisis e interpretación **40%** · Reflexión y registro de IA **20%** ([evaluación](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/docs/evaluacion.md)) |
| **Entrega** | `Archivo → Descargar notebook` (*File → Download notebook*) y súbelo a **e-Aulas** como **`S11_<codigo>.ipynb`** (por ejemplo, `S11_123456.ipynb`) |

**Escenario.** Eres parte del equipo de datos de *Andes Bank* (banco digital ficticio). Un proveedor etiquetó los mensajes de un grupo de intenciones de Banking77 y cometió errores. Tu configuración personal fija **qué intenciones**, **cuántos errores de etiqueta**, **cuántos ensayos de Optuna** puedes pagar y **qué restricciones de despliegue** tiene tu caso de negocio.

### Cómo trabajar
1. En Kaggle, activa **Settings → Internet → On** y deja **Accelerator: None**: todo corre en CPU, que es además el hardware de las restricciones de despliegue.
2. Escribe tu **código estudiantil** y tu nombre en la primera celda de código y ejecuta el notebook **en orden** (*Run All*).
3. Completa las celdas marcadas con ✏️ y responde en las celdas que empiezan con **Tu respuesta:**, citando **tus** números.
4. Antes de entregar, verifica que aparezcan **"Tu configuración personal"** y la **huella de resultados** al final.

| Tarea | Qué haces | Componente de la rúbrica |
|---|---|---|
| 1 · Ejecución | Mejoras tu modelo con **al menos 2 iteraciones documentadas** en una bitácora | Ejecución técnica (40%) |
| 2 · Análisis | Reporte de **robustez** con tus tasas de aprobación | Análisis e interpretación (40%) |
| 3 · Explica y decide | Qué versión desplegar con **tus** restricciones de latencia, exactitud y tamaño | Análisis e interpretación (40%) |
| 4 · Crítica a la IA | Encuentras, explicas y corriges 3 pasos equivocados en un plan de mejora generado por IA | Análisis e interpretación (40%) |
| Reflexión y registro de IA | Al final del notebook | Reflexión y registro de IA (20%) |

In [ ]:
# ✏️ Escribe tu código estudiantil (como aparece en e-Aulas) y tu nombre completo.
STUDENT_ID = ""
STUDENT_NAME = ""

# --- Configuración del curso (no editar) ---
import os, sys, random, subprocess, importlib, hashlib, json
SESSION = "S11"
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules
if FAST_DEV_RUN and not STUDENT_ID.strip():
    STUDENT_ID, STUDENT_NAME = "000000", "Prueba automática"
assert STUDENT_ID.strip(), "⚠️ Escribe tu STUDENT_ID en la primera línea de esta celda y vuelve a ejecutarla."

def ensure(package, import_name=None):
    """Instala un paquete solo si falta."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

PERSONAL_SEED = int(hashlib.sha256(f"{SESSION}:{STUDENT_ID.strip()}".encode()).hexdigest()[:8], 16)
rng = random.Random(PERSONAL_SEED)

def pick(options):
    """Elige una opción de forma determinística a partir de tu código estudiantil."""
    return rng.choice(list(options))

import socket
if os.environ.get("HF_HUB_OFFLINE") != "1":  # los modelos y datos se descargan de Hugging Face
    try:
        socket.create_connection(("huggingface.co", 443), timeout=10).close()
    except OSError:
        raise RuntimeError(
            "Sin conexión a internet: este notebook descarga modelos y datos de Hugging Face.\n"
            "Kaggle: en el panel derecho abre Settings (Session options) y activa Internet. "
            "La opción solo aparece si tu cuenta tiene el teléfono verificado (kaggle.com/settings). "
            "Luego vuelve a ejecutar esta celda. En Colab el internet viene activado. "
            "En tu propio computador: revisa tu conexión, o define HF_HUB_OFFLINE=1 si ya tienes los modelos descargados."
        ) from None

import numpy as np
import torch
random.seed(PERSONAL_SEED); np.random.seed(PERSONAL_SEED % (2**32)); torch.manual_seed(PERSONAL_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sesión {SESSION} | Estudiante: {STUDENT_NAME} ({STUDENT_ID}) | Semilla personal: {PERSONAL_SEED} | Dispositivo: {DEVICE}")
# --- fin de la configuración ---
ensure("optuna")
ensure("datasets")

In [ ]:
INTENT_GROUPS = {
    "tarjetas": ["card_about_to_expire", "card_acceptance", "card_linking", "card_not_working", "card_swallowed",
                 "change_pin", "contactless_not_working", "get_physical_card", "pin_blocked", "virtual_card_not_working"],
    "transferencias": ["beneficiary_not_allowed", "declined_transfer", "failed_transfer", "receiving_money",
                       "transfer_fee_charged", "transfer_into_account", "transfer_timing",
                       "balance_not_updated_after_bank_transfer", "top_up_by_bank_transfer_charge", "cancel_transfer"],
    "recargas_y_cambio": ["automatic_top_up", "pending_top_up", "top_up_limits", "top_up_reverted", "topping_up_by_card",
                          "verify_top_up", "exchange_charge", "exchange_via_app", "fiat_currency_support",
                          "wrong_exchange_rate_for_cash_withdrawal"],
    "cuenta_y_seguridad": ["age_limit", "edit_personal_details", "passcode_forgotten", "terminate_account",
                           "unable_to_verify_identity", "verify_my_identity", "why_verify_identity",
                           "verify_source_of_funds", "lost_or_stolen_phone", "country_support"],
    "cargos_y_reembolsos": ["Refund_not_showing_up", "request_refund", "extra_charge_on_statement",
                            "transaction_charged_twice", "card_payment_fee_charged", "cash_withdrawal_charge",
                            "direct_debit_payment_not_recognised", "pending_card_payment", "reverted_card_payment?",
                            "wrong_amount_of_cash_received"],
}
DEPLOY_SCENARIOS = [
    {"escenario": "Enrutador en vivo del chat de Andes Bank: miles de mensajes por minuto en una CPU compartida",
     "max_latency_ms": 2, "min_accuracy": 0.88, "max_size_mb": 5},
    {"escenario": "Clasificador dentro de la app móvil de Banco Andino: corre en el teléfono, sin conexión",
     "max_latency_ms": 25, "min_accuracy": 0.90, "max_size_mb": 75},
    {"escenario": "Asistente para agentes del call center de NovaTel Pagos: servidor dedicado",
     "max_latency_ms": 60, "min_accuracy": 0.92, "max_size_mb": 200},
    {"escenario": "Clasificación nocturna por lotes de los correos de la Cooperativa Horizonte",
     "max_latency_ms": 500, "min_accuracy": 0.93, "max_size_mb": 1000},
]
CONFIG = {
    "noise_rate": pick([0.05, 0.1, 0.2]),
    "intent_group": pick(sorted(INTENT_GROUPS)),
    "budget_trials": pick([10, 15, 20]),
    "deploy_constraints": pick(DEPLOY_SCENARIOS),
}
CONFIG["intents"] = INTENT_GROUPS[CONFIG["intent_group"]]

print("🎯 Tu configuración personal")
print(f"  Grupo de intenciones: {CONFIG['intent_group']} ({len(CONFIG['intents'])} intenciones)")
print(f"  Tasa de errores de etiqueta del proveedor: {CONFIG['noise_rate']:.0%}")
print(f"  Presupuesto de Optuna: {CONFIG['budget_trials']} ensayos")
dc = CONFIG["deploy_constraints"]
print(f"  Caso de despliegue: {dc['escenario']}")
print(f"  Restricciones: latencia ≤ {dc['max_latency_ms']} ms por mensaje · exactitud ≥ {dc['min_accuracy']:.2f} · tamaño ≤ {dc['max_size_mb']} MB")

## Tarea 1 — Mejora tu modelo con al menos 2 iteraciones documentadas (Ejecución técnica)

1. Ejecuta la línea base (**E0**) y lee el análisis de errores por segmento.
2. Escribe en `HIPOTESIS_1` y `HIPOTESIS_2` hipótesis que salgan de **tus** números (intención, par confundido o segmento de longitud).
3. Corre **al menos dos iteraciones**. Por defecto, la iteración 1 busca etiquetas sospechosas con probabilidades de validación cruzada (como en el lab) y la iteración 2 busca hiperparámetros con Optuna usando **tu** presupuesto de ensayos. Puedes cambiar la acción de la iteración 1 (`"descartar"` o `"reetiquetar"`) o agregar una iteración 3 propia.
4. La regla de decisión ya está escrita **antes** de experimentar: una iteración se conserva solo si mejora el macro-F1 de validación en al menos `MIN_GAIN` frente a la versión campeona. **El conjunto de prueba no se usa hasta la Tarea 3.**

In [ ]:
import time, string, tempfile, warnings, statistics, pickle
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
pd.set_option("display.max_colwidth", 90)

INTENTS = CONFIG["intents"]
MAX_PER_INTENT = 60 if FAST_DEV_RUN else None      # la prueba automática usa menos datos, mismo código
MAX_TEST_PER_INTENT = 20 if FAST_DEV_RUN else None
N_TRIALS = 3 if FAST_DEV_RUN else CONFIG["budget_trials"]
N_BOOTSTRAP = 200 if FAST_DEV_RUN else 1000
N_LATENCY = 10 if FAST_DEV_RUN else 50
N_SEEDS = 3 if FAST_DEV_RUN else 20
SPLIT_SEED = 42                                      # la partición es fija; el ruido depende de tu semilla personal

banking = load_dataset("legacy-datasets/banking77")
LABEL_NAMES = banking["train"].features["label"].names

def to_frame(split, max_per_intent=None):
    df = split.to_pandas()
    df["intent"] = [LABEL_NAMES[i] for i in df["label"]]
    df = df[df["intent"].isin(INTENTS)][["text", "intent"]]
    if max_per_intent:
        df = df.sample(frac=1, random_state=SPLIT_SEED).groupby("intent").head(max_per_intent)
    return df.reset_index(drop=True)

pool_df = to_frame(banking["train"], MAX_PER_INTENT)
test_df = to_frame(banking["test"], MAX_TEST_PER_INTENT)
train_df, valid_df = train_test_split(pool_df, test_size=0.2, stratify=pool_df["intent"], random_state=SPLIT_SEED)
train_df, valid_df = train_df.reset_index(drop=True), valid_df.reset_index(drop=True)

# El "proveedor" se equivoca en noise_rate de las etiquetas de entrenamiento (con tu semilla personal).
noise_rng = np.random.default_rng(PERSONAL_SEED)
flip = noise_rng.choice(len(train_df), size=int(round(CONFIG["noise_rate"] * len(train_df))), replace=False)
labels = train_df["intent"].to_numpy(dtype=object).copy()
for i in flip:
    labels[i] = noise_rng.choice([c for c in INTENTS if c != labels[i]])
train_df["label"] = labels                                   # lo único que ve el modelo
train_df["is_flipped"] = train_df["label"] != train_df["intent"]   # solo para auditar (imposible en la vida real)
print(f"Entrenamiento {len(train_df)} (etiquetas erróneas: {train_df['is_flipped'].sum()}) · validación {len(valid_df)} · prueba {len(test_df)}")

In [ ]:
# Herramientas del lab (no necesitas modificarlas).
def make_model(C=10.0, ngram_max=1, analyzer="word"):
    low = 1 if analyzer == "word" else 2
    return make_pipeline(TfidfVectorizer(analyzer=analyzer, ngram_range=(low, ngram_max), sublinear_tf=True),
                         LogisticRegression(C=C, max_iter=3000))

def evaluate(model, df):
    pred = model.predict(df["text"])
    return {"macro_f1": f1_score(df["intent"], pred, average="macro"), "accuracy": accuracy_score(df["intent"], pred)}

MIN_GAIN = 0.01
LOG, MODELS, TRAIN_SETS = [], {}, {}

def champion():
    return [r["id"] for r in LOG if r["decision"] in ("baseline", "keep")][-1]

def log_experiment(exp_id, hypothesis, change, model, train_texts, train_labels, notes=""):
    m = evaluate(model, valid_df)
    if LOG:
        best = next(r for r in LOG if r["id"] == champion())["val_macro_f1"]
        delta = m["macro_f1"] - best
        decision = "keep" if delta >= MIN_GAIN else "discard"
    else:
        delta, decision = 0.0, "baseline"
    LOG.append({"id": exp_id, "hypothesis": hypothesis, "change": change,
                "val_macro_f1": round(m["macro_f1"], 4), "val_accuracy": round(m["accuracy"], 4),
                "delta_vs_champion": round(delta, 4), "decision": decision, "notes": notes})
    MODELS[exp_id], TRAIN_SETS[exp_id] = model, (np.asarray(train_texts, dtype=object), np.asarray(train_labels, dtype=object))
    return pd.DataFrame(LOG)

def paired_bootstrap_ci(y_true, pred_a, pred_b, n=N_BOOTSTRAP, seed=PERSONAL_SEED):
    """Intervalo del 95% para macro-F1(b) − macro-F1(a) con los mismos mensajes remuestreados."""
    y_true, pred_a, pred_b = map(np.asarray, (y_true, pred_a, pred_b))
    boot = np.random.default_rng(seed)
    deltas = []
    for _ in range(n):
        idx = boot.integers(0, len(y_true), len(y_true))
        deltas.append(f1_score(y_true[idx], pred_b[idx], average="macro") - f1_score(y_true[idx], pred_a[idx], average="macro"))
    return np.percentile(deltas, [2.5, 97.5])

In [ ]:
# Línea base E0 y análisis de errores por segmento (validación).
baseline = make_model().fit(train_df["text"], train_df["label"])
log_experiment("E0", "—", "Línea base: unigramas, C=10, etiquetas del proveedor", baseline, train_df["text"], train_df["label"])

val_pred = baseline.predict(valid_df["text"])
per_intent = pd.DataFrame({"f1": f1_score(valid_df["intent"], val_pred, average=None, labels=INTENTS),
                           "train_examples": train_df["label"].value_counts().reindex(INTENTS).to_numpy()},
                          index=INTENTS).sort_values("f1").round(3)
cm = confusion_matrix(valid_df["intent"], val_pred, labels=INTENTS)
confused = pd.DataFrame([(INTENTS[i], INTENTS[j], cm[i, j]) for i in range(len(INTENTS)) for j in range(len(INTENTS))
                         if i != j and cm[i, j] > 0], columns=["true intent", "predicted as", "count"])
lengths = valid_df.assign(correct=val_pred == valid_df["intent"],
                          bucket=valid_df["text"].map(lambda t: "corto (≤ 6)" if len(t.split()) <= 6 else ("medio (7–14)" if len(t.split()) <= 14 else "largo (≥ 15)")))
print("F1 por intención (las 4 más débiles):"); print(per_intent.head(4).to_string())
print("\nPares más confundidos:"); print(confused.sort_values("count", ascending=False).head(5).to_string(index=False))
print("\nExactitud por longitud (palabras):"); print(lengths.groupby("bucket")["correct"].agg(["size", "mean"]).round(3).to_string())

In [ ]:
# ✏️ TU CÓDIGO AQUÍ — Iteración 1: etiquetas sospechosas (inspirado en confident learning).
HIPOTESIS_1 = "✏️ Escribe tu hipótesis: qué error de E0 esperas corregir y por qué (cita tus números)."
ACCION_1 = "descartar"          # ✏️ "descartar" o "reetiquetar"

label_arr = train_df["label"].to_numpy(dtype=object)
proba = cross_val_predict(make_model(), train_df["text"], label_arr,
                          cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SPLIT_SEED), method="predict_proba")
classes = np.unique(label_arr)
given = np.searchsorted(classes, label_arr.astype(str))
self_conf = proba[np.arange(len(proba)), given]
thresholds = np.array([self_conf[given == j].mean() for j in range(len(classes))])
above = proba >= thresholds
suggested = np.where(above, proba, -1).argmax(axis=1)
flagged = above.any(axis=1) & (suggested != given)

tp = int((flagged & train_df["is_flipped"].to_numpy()).sum())
det_precision = tp / max(int(flagged.sum()), 1)
det_recall = tp / max(int(train_df["is_flipped"].sum()), 1)
print(f"Marcados: {flagged.sum()} · precisión del detector {det_precision:.2f} · recall {det_recall:.2f} (auditoría posible solo en simulación)")

if ACCION_1 == "reetiquetar":
    it1_texts, it1_labels = train_df["text"].to_numpy(dtype=object), np.where(flagged, classes[suggested], label_arr)
else:
    it1_texts, it1_labels = train_df["text"].to_numpy(dtype=object)[~flagged], label_arr[~flagged]
model_it1 = make_model().fit(it1_texts, it1_labels)
log_experiment("E1", HIPOTESIS_1, f"Etiquetas sospechosas (CV 5 pliegues): {ACCION_1} {flagged.sum()} ejemplos",
               model_it1, it1_texts, it1_labels)

In [ ]:
# ✏️ TU CÓDIGO AQUÍ — Iteración 2: búsqueda de hiperparámetros con Optuna (tu presupuesto de ensayos).
HIPOTESIS_2 = "✏️ Escribe tu hipótesis: por qué la configuración de la línea base (unigramas, C=10) no sería la mejor para tus datos."

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
opt_texts, opt_labels = TRAIN_SETS[champion()]

def objective(trial):
    analyzer = trial.suggest_categorical("analyzer", ["word", "char_wb"])
    ngram_max = trial.suggest_int("word_ngram_max", 1, 3) if analyzer == "word" else trial.suggest_int("char_ngram_max", 3, 5)
    C = trial.suggest_float("C", 0.1, 100, log=True)
    model = make_model(C=C, ngram_max=ngram_max, analyzer=analyzer).fit(opt_texts, opt_labels)
    return evaluate(model, valid_df)["macro_f1"]          # validación: la prueba no entra aquí

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=PERSONAL_SEED % (2**32)))
study.optimize(objective, n_trials=N_TRIALS)
bp = study.best_params
best_ngram = bp.get("word_ngram_max", bp.get("char_ngram_max"))
model_it2 = make_model(C=bp["C"], ngram_max=best_ngram, analyzer=bp["analyzer"]).fit(opt_texts, opt_labels)
log_experiment("E2", HIPOTESIS_2, f"Optuna TPE ({N_TRIALS} ensayos) sobre los datos de {champion()}: {bp['analyzer']}, n-gramas hasta {best_ngram}, C={bp['C']:.2f}",
               model_it2, opt_texts, opt_labels, notes="elegido en validación")

In [ ]:
# ✏️ (Opcional) Iteración 3 propia: cambia HACER_ITERACION_3 a True y edita el cambio.
HACER_ITERACION_3 = False
if HACER_ITERACION_3:
    t3, y3 = TRAIN_SETS[champion()]
    model_it3 = make_model(C=30, ngram_max=5, analyzer="char_wb").fit(t3, y3)   # ✏️ tu cambio
    log_experiment("E3", "✏️ tu hipótesis", "✏️ describe tu cambio", model_it3, t3, y3)

log_df = pd.DataFrame(LOG)
lo, hi = paired_bootstrap_ci(valid_df["intent"], baseline.predict(valid_df["text"]), MODELS[champion()].predict(valid_df["text"]))
print(f"Campeón: {champion()} · IC 95% de la ganancia en macro-F1 frente a E0 (validación): [{lo:+.4f}, {hi:+.4f}]")
log_df

**Tu respuesta:** *(5–8 líneas)* ¿Qué iteración mejoró y cuál no? ¿La ganancia del campeón es mayor que el ruido de la evaluación (mira el intervalo)? Relaciona la precisión y el recall del detector con tu tasa de ruido. ¿Qué probarías en la siguiente iteración y por qué?

## Tarea 2 — Reporte de robustez (Análisis e interpretación)

Corre las pruebas de invariancia (INV) estilo CheckList sobre la línea base y tu campeón. La tasa de aprobación es la fracción de mensajes cuya predicción **no cambia** con la perturbación.

In [ ]:
IRRELEVANT = ["By the way, I love your app.", "I'm writing from my phone.", "Thanks in advance!",
              "It is raining here today.", "Sorry for my English."]

def add_typo(text, rnd):
    words = text.split()
    spots = [i for i, w in enumerate(words) if len(w) >= 4]
    if not spots:
        return text
    i = rnd.choice(spots)
    w = words[i]
    j = rnd.randrange(len(w) - 1)
    words[i] = w[:j] + w[j + 1] + w[j] + w[j + 2:]
    return " ".join(words)

PERTURBATIONS = {"typo": add_typo, "UPPERCASE": lambda t, rnd: t.upper(),
                 "irrelevant sentence": lambda t, rnd: f"{t.rstrip()} {rnd.choice(IRRELEVANT)}",
                 "no punctuation": lambda t, rnd: t.translate(str.maketrans("", "", string.punctuation))}

def invariance_report(predict, texts, seed=PERSONAL_SEED):
    original = np.asarray(predict(list(texts)))
    rates, failures = {}, {}
    for name, perturb in PERTURBATIONS.items():
        rnd = random.Random(seed)
        changed = [perturb(t, rnd) for t in texts]
        same = np.asarray(predict(changed)) == original
        rates[name] = round(float(same.mean()), 4)
        failures[name] = [(changed[i], original[i]) for i in np.where(~same)[0][:3]]
    return rates, failures

inv_texts = valid_df["text"].tolist()
ROBUSTNESS = {}
ROBUSTNESS["E0"], _ = invariance_report(baseline.predict, inv_texts)
ROBUSTNESS[champion()], champ_failures = invariance_report(MODELS[champion()].predict, inv_texts)
display_rob = pd.DataFrame(ROBUSTNESS).rename_axis("INV test")
print(display_rob.to_string())
for name, examples in champ_failures.items():
    for text, before in examples[:2]:
        print(f"  ✗ [{name}] {text!r} (antes: {before})")

**Tu respuesta:** *(5–8 líneas)*
1. ¿Qué perturbación hace fallar más a tu campeón? Cita las tasas de E0 y del campeón.
2. Explica **uno** de los ejemplos fallidos: ¿por qué crees que el modelo cambió de opinión?
3. Propón **una** corrección y cómo verificarías que no empeora la exactitud.
4. ¿Por qué la negación ("I want to cancel my transfer" → "I don't want to cancel my transfer") **no** es una prueba de invariancia para un clasificador de intenciones?

## Tarea 3 — Explica y decide: ¿qué versión despliegas? (Análisis e interpretación)

Comparamos cuatro candidatas en **CPU**: la línea base, tu campeón TF-IDF, y un clasificador sobre *embeddings* de `sentence-transformers/all-MiniLM-L6-v2` (en float32 y cuantizado a int8 con cuantización dinámica), entrenado con los datos de tu campeón. Medimos en validación la exactitud (con su intervalo del 95%), la latencia por mensaje (mediana), el tamaño en disco y la robustez promedio. La tabla marca qué candidatas cumplen **tus** restricciones.

In [ ]:
from transformers import AutoTokenizer, AutoModel
warnings.filterwarnings("ignore", message=r".*quantiz")   # avisos de obsolescencia de la API de cuantización

ENC_NAME = "sentence-transformers/all-MiniLM-L6-v2"
enc_tok = AutoTokenizer.from_pretrained(ENC_NAME)
enc_fp32 = AutoModel.from_pretrained(ENC_NAME).eval()     # en CPU: el hardware de las restricciones

def quantize_int8(model):
    """Cuantización dinámica int8 de las capas nn.Linear; None si esta versión de PyTorch no la tiene."""
    engines = torch.backends.quantized.supported_engines
    if torch.backends.quantized.engine not in engines or torch.backends.quantized.engine == "none":
        for engine in ("x86", "fbgemm", "qnnpack"):
            if engine in engines:
                torch.backends.quantized.engine = engine
                break
    try:
        from torch.ao.quantization import quantize_dynamic
    except ImportError:
        try:
            from torch.quantization import quantize_dynamic
        except ImportError:
            print("Esta versión de PyTorch no incluye quantize_dynamic (migró a torchao): se omite la versión int8.")
            return None
    try:
        return quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)
    except Exception as exc:
        print(f"La cuantización dinámica no está disponible aquí ({type(exc).__name__}): se omite la versión int8.")
        return None

def embed(encoder, texts, batch_size=64):
    out = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch_size):
            b = enc_tok(list(texts[i:i + batch_size]), padding=True, truncation=True, max_length=64, return_tensors="pt")
            h = encoder(**b).last_hidden_state
            mask = b["attention_mask"].unsqueeze(-1).float()
            out.append(torch.nn.functional.normalize((h * mask).sum(1) / mask.sum(1), dim=-1).numpy())
    return np.vstack(out)

class EmbeddingClassifier:
    """Encoder congelado + regresión logística: expone predict(textos) como un pipeline de sklearn."""
    def __init__(self, encoder):
        self.encoder, self.head = encoder, LogisticRegression(C=10.0, max_iter=3000)
    def fit(self, texts, labels):
        self.head.fit(embed(self.encoder, list(texts)), labels)
        return self
    def predict(self, texts):
        return self.head.predict(embed(self.encoder, list(texts)))

def size_mb(candidate):
    if isinstance(candidate, EmbeddingClassifier):
        with tempfile.TemporaryDirectory() as tmp:
            torch.save(candidate.encoder.state_dict(), os.path.join(tmp, "enc.pt"))
            return (os.path.getsize(os.path.join(tmp, "enc.pt")) + len(pickle.dumps(candidate.head))) / 1e6
    return len(pickle.dumps(candidate)) / 1e6

def latency_ms(candidate, texts):
    candidate.predict(texts[:1])                              # calentamiento
    times = []
    for t in texts:
        start = time.perf_counter()
        candidate.predict([t])
        times.append((time.perf_counter() - start) * 1000)
    return statistics.median(times)

def wilson_low(k, n, z=1.96):
    p = k / n
    return (p + z**2 / (2 * n) - z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / (1 + z**2 / n)

champ_texts, champ_labels = TRAIN_SETS[champion()]
CANDIDATES = {"E0 TF-IDF (línea base)": baseline}
if champion() != "E0":
    CANDIDATES[f"{champion()} TF-IDF (campeón)"] = MODELS[champion()]
CANDIDATES["MiniLM + LR (float32)"] = EmbeddingClassifier(enc_fp32).fit(champ_texts, champ_labels)
enc_int8 = quantize_int8(enc_fp32)
if enc_int8 is not None:
    CANDIDATES["MiniLM + LR (int8)"] = EmbeddingClassifier(enc_int8).fit(champ_texts, champ_labels)

lat_texts = valid_df["text"].sample(N_LATENCY, random_state=SPLIT_SEED).tolist()
rows = []
for name, cand in CANDIDATES.items():
    pred = cand.predict(valid_df["text"].tolist())
    k, n = int((pred == valid_df["intent"].to_numpy()).sum()), len(valid_df)
    rob = ROBUSTNESS.get(name.split()[0]) if "TF-IDF" in name else None
    if rob is None:
        rob, _ = invariance_report(cand.predict, inv_texts)
        ROBUSTNESS[name] = rob
    rows.append({"version": name, "val_accuracy": round(k / n, 4), "acc_ci95_low": round(float(wilson_low(k, n)), 4),
                 "val_macro_f1": round(f1_score(valid_df["intent"], pred, average="macro"), 4),
                 "latency_ms": round(latency_ms(cand, lat_texts), 3), "size_mb": round(size_mb(cand), 2),
                 "robustness_mean": round(float(np.mean(list(rob.values()))), 4)})
table = pd.DataFrame(rows).set_index("version")
table["cumple"] = ((table["latency_ms"] <= dc["max_latency_ms"]) & (table["val_accuracy"] >= dc["min_accuracy"])
                   & (table["size_mb"] <= dc["max_size_mb"]))
table["cumple_con_margen"] = table["cumple"] & (table["acc_ci95_low"] >= dc["min_accuracy"])
print(f"Restricciones: latencia ≤ {dc['max_latency_ms']} ms · exactitud ≥ {dc['min_accuracy']} · tamaño ≤ {dc['max_size_mb']} MB")
table

In [ ]:
# ✏️ TU DECISIÓN: escribe el nombre exacto de la versión que despliegas (o "ninguna").
# Por defecto se elige la candidata que cumple con mayor exactitud: ¿estás de acuerdo con esa regla?
factibles = table[table["cumple"]]
ELEGIDA = factibles["val_accuracy"].idxmax() if len(factibles) else "ninguna"
print("Versión elegida:", ELEGIDA)

# Reporte final en PRUEBA: una sola vez, para todas las candidatas (ninguna decisión sale de esta tabla).
test_rows = []
for name, cand in CANDIDATES.items():
    pred = cand.predict(test_df["text"].tolist())
    test_rows.append({"version": name, "test_accuracy": round(accuracy_score(test_df["intent"], pred), 4),
                      "test_macro_f1": round(f1_score(test_df["intent"], pred, average="macro"), 4)})
test_table = pd.DataFrame(test_rows).set_index("version")
test_table.join(table[["val_accuracy", "cumple"]])

**Tu respuesta:** *(memo de 6–10 líneas para el comité de Andes Bank)*
- ¿Qué versión despliegas y por qué? Cita **tus** números de exactitud, latencia y tamaño frente a **tus** restricciones.
- ¿La cumple *con margen* (límite inferior del intervalo) o por muy poco? ¿Qué riesgo implica?
- ¿Qué ganas y qué pierdes frente a la segunda mejor opción (robustez, costo, mantenimiento)?
- Si ninguna cumple: ¿qué le propones al negocio (relajar una restricción, otra iteración, más datos, otro hardware)?
- ¿Qué señal monitorearías en producción durante el primer mes?

## Tarea 4 — Crítica a la IA (Análisis e interpretación)

Un compañero le pidió a un asistente de IA "un plan para mejorar el clasificador de intenciones antes del comité". Esta es la respuesta:

> **Plan de mejora (generado por IA)**
> 1. Registra cada experimento en una bitácora con la versión de los datos, la configuración y la semilla.
> 2. Para elegir `C` y el rango de n-gramas, corre Optuna con el macro-F1 del **conjunto de prueba** como objetivo: así optimizas directamente la métrica que vas a reportar.
> 3. Calcula la autoconfianza de cada ejemplo con validación cruzada y **elimina todos los ejemplos con probabilidad menor a 0.5** para su etiqueta: son ejemplos difíciles que solo confunden al modelo.
> 4. Mide la latencia en el mismo tipo de servidor en el que vas a desplegar.
> 5. Entrena el modelo final con **20 semillas distintas y reporta la mejor exactitud** como el resultado del modelo; así muestras su verdadero potencial.
> 6. Antes de desplegar, corre las pruebas de invariancia y compáralas con las de la versión anterior.
>
> ```python
> def objective(trial):
>     C = trial.suggest_float("C", 0.01, 100, log=True)
>     model = make_model(C=C).fit(train_texts, train_labels)
>     return evaluate(model, test_df)["macro_f1"]
>
> hard = self_conf < 0.5
> train_texts, train_labels = train_texts[~hard], train_labels[~hard]
>
> scores = [train_and_score(seed) for seed in range(20)]
> print("Exactitud del modelo:", max(scores))
> ```

El plan tiene **3 pasos equivocados**. La celda siguiente te da evidencia con **tus** datos para dos de ellos.

In [ ]:
# Evidencia 1: ¿qué pasa si eliminamos TODO lo que tiene autoconfianza < 0.5?
hard = self_conf < 0.5
dropped_ok = int((hard & ~train_df["is_flipped"].to_numpy()).sum())
share_by_intent = pd.Series(hard, index=train_df["label"]).groupby(level=0).mean().sort_values(ascending=False)
model_hard = make_model().fit(train_df["text"].to_numpy(dtype=object)[~hard], label_arr[~hard])
print(f"Umbral fijo 0.5: se eliminan {hard.sum()} ejemplos; {dropped_ok} tenían la etiqueta CORRECTA.")
print(f"Intención más recortada: {share_by_intent.index[0]} ({share_by_intent.iloc[0]:.0%} de sus ejemplos)")
print(f"Macro-F1 de validación: umbral 0.5 = {evaluate(model_hard, valid_df)['macro_f1']:.4f} · campeón = {evaluate(MODELS[champion()], valid_df)['macro_f1']:.4f}")

# Evidencia 2: la receta de la línea base (con tu tasa de ruido) repetida con distintas semillas de partición y de ruido.
seed_scores = []
for s in range(N_SEEDS):
    tr, va = train_test_split(pool_df, test_size=0.2, stratify=pool_df["intent"], random_state=s)
    ruido = np.random.default_rng(PERSONAL_SEED + s)
    y_tr = tr["intent"].to_numpy(dtype=object).copy()
    for i in ruido.choice(len(y_tr), size=int(round(CONFIG["noise_rate"] * len(y_tr))), replace=False):
        y_tr[i] = ruido.choice([c for c in INTENTS if c != y_tr[i]])
    seed_scores.append(accuracy_score(va["intent"], make_model().fit(tr["text"], y_tr).predict(va["text"])))
seed_scores = np.array(seed_scores)
print(f"\n{N_SEEDS} semillas · exactitud media {seed_scores.mean():.4f} ± {seed_scores.std(ddof=1):.4f} (desv. est.) · "
      f"mínima {seed_scores.min():.4f} · máxima {seed_scores.max():.4f}")

**Tu respuesta:** completa la tabla (una fila por cada paso equivocado; usa los números de la celda anterior donde apliquen).

| Paso del plan | ¿Por qué es un error? | ¿Cómo lo corriges? |
|---|---|---|
|  |  |  |
|  |  |  |
|  |  |  |

> 🧩 **Hito H4 del proyecto final (grupal, ya entregado):** la iteración de mejora del modelo de su proyecto se entregó el viernes 23 de octubre a las 23:59, en la bitácora de la sección 5 del informe técnico (hipótesis → cambio → resultado → decisión), junto con la entrega final. Hoy no se entrega nada del proyecto: revisen esa iteración con la misma disciplina de este challenge (decidir con validación y no tocar la prueba) y preparen cómo contarla en el *pitch* de la S12, que empieza hoy a las 10:00. Detalles en el [README del proyecto final](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/proyecto-final/README.md).

In [ ]:
def r4(x):
    return None if x is None else round(float(x), 4)

chosen_test = test_table.loc[ELEGIDA] if ELEGIDA in test_table.index else None
RESULTS = {
    "val_macro_f1_base": r4(LOG[0]["val_macro_f1"]),
    "val_macro_f1_campeon": r4(next(r for r in LOG if r["id"] == champion())["val_macro_f1"]),
    "campeon": champion(),
    "n_iteraciones": len(LOG) - 1,
    "ganancia_ic95": [r4(lo), r4(hi)],
    "deteccion_precision": r4(det_precision),
    "deteccion_recall": r4(det_recall),
    "inv_typo_base": r4(ROBUSTNESS["E0"]["typo"]),
    "inv_typo_campeon": r4(ROBUSTNESS[champion()]["typo"]),
    "inv_irrelevante_campeon": r4(ROBUSTNESS[champion()]["irrelevant sentence"]),
    "version_elegida": ELEGIDA,
    "elegida_val_accuracy": r4(table.loc[ELEGIDA, "val_accuracy"]) if ELEGIDA in table.index else None,
    "elegida_tamano_mb": r4(table.loc[ELEGIDA, "size_mb"]) if ELEGIDA in table.index else None,
    "elegida_test_macro_f1": r4(chosen_test["test_macro_f1"]) if chosen_test is not None else None,
    "semillas_media": r4(seed_scores.mean()),
    "semillas_max": r4(seed_scores.max()),
}
RESULTS

## 🪞 Reflexión y registro de uso de IA (20%)

### Reflexión (responde en 3–5 líneas cada una)
1. ¿Qué aprendiste hoy que puedas aplicar en tu trabajo u organización?
2. ¿Qué fue lo más difícil y cómo lo resolviste?
3. Autoevaluación: ¿qué nota (0.0–5.0) te pondrías en este challenge y por qué?

### 🤖 Registro de uso de IA (obligatorio)
| # | Herramienta (ChatGPT, Claude, Codex, Copilot…) | ¿Para qué la usaste? | Prompt principal (resumido) | ¿Qué verificaste o corregiste de su respuesta? |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |

> Si no usaste asistentes de IA, escribe: *"No usé asistentes de IA en este challenge."* Recuerda: puedes usar IA, pero **eres responsable de todo lo que entregas** y podrías ser seleccionado(a) para una micro-sustentación.

In [ ]:
# --- Huella de resultados (no editar) ---
def results_fingerprint(results):
    payload = json.dumps({"session": SESSION, "student": STUDENT_ID.strip(), "results": results},
                         sort_keys=True, default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode()).hexdigest()[:12]

print(json.dumps({"CONFIG": CONFIG, "RESULTS": RESULTS}, indent=2, ensure_ascii=False, default=str))
print("🔏 Huella de resultados:", results_fingerprint({"CONFIG": CONFIG, "RESULTS": RESULTS}))